# **Notebook 5: Solution V1 — RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` — Created by Notebooks 3+4
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `v1_metrics.csv` — Per-row Baseline vs V1 scores _(Evidence for comparative analysis)_

---

### **Task 3.3: Evaluate Solution V1**

> This task is split into five measurements (3.3.1–3.3.5). Run the shared setup cell below first (it loads the model, ChromaDB, and test data), then work through each measurement.

**── Shared setup ──**
Load the base model, reload ChromaDB (same embedding model as NB4), and load `df_test.csv` + `outputs.json`. Define helper functions `generate_baseline()` and `generate_naive_rag()` here so every subtask below can reuse them.

In [11]:
%pip install -q tqdm rouge-score nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch, json
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
device = "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to(device)
model.eval()

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectordb = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

df_test = pd.read_csv("df_test.csv")
with open("outputs.json") as f:
    outputs = json.load(f)

def _run_model(messages):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def generate_baseline(query):
    messages = [{"role": "user", "content": query}]
    return _run_model(messages)

def generate_naive_rag(query):
    top_doc = vectordb.similarity_search(query, k=1)[0].page_content
    messages = [
        {"role": "system", "content": f"Answer strictly using this SOP:\n{top_doc}"},
        {"role": "user", "content": query},
    ]
    return _run_model(messages)


print("Device:", device)
print("Chroma doc count:", vectordb._collection.count())
print("Test rows:", len(df_test))
print("outputs.json keys:", list(outputs.keys()))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6675.84it/s]
/var/folders/96/jylj5rkx5n13cx5tm6mgdjc80000gn/T/ipykernel_3191/2369118868.py:15: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)


Device: mps
Chroma doc count: 13
Test rows: 391
outputs.json keys: ['test_query', 'ground_truth', 'baseline_output', 'naive_rag_output']


#### **3.3.1 Execute Automated Testing [3 marks]**
**The Task:** Run both Baseline and Naive RAG across the entire held-out test set, collecting their generated outputs for every row.

**Hints & Tips:**
* Loop over `df_test` rows; for each query call both `generate_baseline()` and `generate_naive_rag()`.
* Store raw outputs in a list of dicts so the later measurements can score them.
* This is the most time-consuming cell — if constrained, `df_test.sample(50)` is acceptable.

**Learner Inference:** Automated testing across the full set gives statistically meaningful results, not a single cherry-picked query.

In [ ]:
# YOUR CODE HERE
from tqdm import tqdm

N = 50
eval_df = df_test.sample(N, random_state=42).reset_index(drop=True)

results = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    query = row["instruction"]

    baseline_out = generate_baseline(query)
    rag_out = generate_naive_rag(query)
    retrieved_sop = vectordb.similarity_search(query, k=1)[0].page_content

    results.append({
        "query": query,
        "baseline_output": baseline_out,
        "rag_output": rag_out,
        "retrieved_sop": retrieved_sop,
    })

print("Collected:", len(results), "rows")
print("\nSample query:", results[0]["query"][:80])
print("Baseline:", results[0]["baseline_output"][:120])
print("RAG:", results[0]["rag_output"][:120])

100%|██████████| 50/50 [09:29<00:00, 11.39s/it]

Collected: 50 rows

Sample query: I cannot afford order {{Order Number}}
Baseline: I'm sorry to hear that you can't afford the order with Order Number {{Order Number}}. It's important to ensure that all 
RAG: We apologize for any inconvenience caused. To proceed, we need your assistance to resolve the payment issue associated w


In [7]:
with open("v1_raw_results.json", "w") as f:
    json.dump(results, f)
print("Saved", len(results), "rows to v1_raw_results.json")

Saved 50 rows to v1_raw_results.json


#### **3.3.2 Measure Format Adherence [2 marks]**
**The Task:** Validate the syntactic correctness of the generated outputs and report the adherence rate.

**Hints & Tips:**
* For the baseline/RAG free-text responses, "format adherence" means the output is well-formed and non-empty (the strict JSON check applies mainly to the fine-tuned router in NB7).
* Report the percentage of outputs that parsed/validated successfully.

**Learner Inference:** Format adherence tells you how often the system produces usable output before you even check correctness.

In [8]:
# YOUR CODE HERE
def is_well_formed(text):
    """Well-formed = non-empty, not just whitespace, has real content."""
    return isinstance(text, str) and len(text.strip()) > 0

def format_adherence_rate(results, key):
    passed = sum(is_well_formed(r[key]) for r in results)
    return passed / len(results) * 100

baseline_fa = format_adherence_rate(results, "baseline_output")
rag_fa      = format_adherence_rate(results, "rag_output")

print(f"Format Adherence — Baseline : {baseline_fa:.1f}%")
print(f"Format Adherence — Naive RAG: {rag_fa:.1f}%")

Format Adherence — Baseline : 100.0%
Format Adherence — Naive RAG: 100.0%


#### **3.3.3 Measure Execution Success (ROUGE/BLEU) [2 marks]**
**The Task:** Evaluate semantic similarity of each output against SOP-grounded references using ROUGE-1, ROUGE-L, and BLEU.

**Hints & Tips:**
* Use SOP-grounded references — retrieve the correct SOP per test row so policy-specific language is rewarded.
* Generic references falsely reward vague baseline answers — avoid them.
* `rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)` and `sentence_bleu` with `SmoothingFunction().method1`.

**Learner Inference:** ROUGE/BLEU measure how close the output is to a correct, policy-grounded answer.

In [12]:
# YOUR CODE HERE
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

def score_one(prediction, reference):
    """Score a single answer against its SOP reference."""
    r = scorer.score(reference, prediction)          # ROUGE
    bleu = sentence_bleu(
        [reference.split()], prediction.split(),      # BLEU
        smoothing_function=smooth
    )
    return {
        "rouge1": r["rouge1"].fmeasure,
        "rougeL": r["rougeL"].fmeasure,
        "bleu":   bleu,
    }

def average_scores(results, key):
    scored = [score_one(r[key], r["retrieved_sop"]) for r in results]
    return {
        "rouge1": sum(s["rouge1"] for s in scored) / len(scored),
        "rougeL": sum(s["rougeL"] for s in scored) / len(scored),
        "bleu":   sum(s["bleu"]   for s in scored) / len(scored),
    }

baseline_scores = average_scores(results, "baseline_output")
rag_scores      = average_scores(results, "rag_output")

print("Metric   | Baseline | Naive RAG")
print("-" * 38)
for m in ["rouge1", "rougeL", "bleu"]:
    print(f"{m:8} | {baseline_scores[m]:.4f}   | {rag_scores[m]:.4f}")

Metric   | Baseline | Naive RAG
--------------------------------------
rouge1   | 0.1772   | 0.1916
rougeL   | 0.0810   | 0.1033
bleu     | 0.0020   | 0.0064


#### **3.3.4 Measure Output Consistency [1 mark]**
**The Task:** Evaluate deterministic behaviour by running the same query multiple times under `do_sample=False` and confirming identical outputs.

**Hints & Tips:**
* Run the same query 3 times; assert all outputs are identical.
* With `do_sample=False, temperature=None`, greedy decoding should be fully deterministic.

**Learner Inference:** Deterministic inference means your evaluation is reproducible — the same input always gives the same output.

In [13]:
# YOUR CODE HERE
test_q = results[0]["query"]

runs = [generate_naive_rag(test_q) for _ in range(3)]

all_identical = all(r == runs[0] for r in runs)

print("Query:", test_q[:70])
print("All 3 runs identical:", all_identical)
print("Consistency Rate: {:.1f}%".format(100.0 if all_identical else 0.0))

Query: I cannot afford order {{Order Number}}
All 3 runs identical: True
Consistency Rate: 100.0%


#### **3.3.5 Measure Hallucination Frequency [2 marks]**
**The Task:** Evaluate how often outputs contain unsupported claims, invalid references, missing functionality, or policy violations.

**Hints & Tips:**
* Compare outputs against the retrieved SOP — flag any specific claim (dates, numbers, policies) not supported by the context.
* Report hallucination frequency as a percentage for both Baseline and Naive RAG.

**Learner Inference:** This quantifies the core problem RAG is meant to solve — grounding responses to reduce fabrication.

In [14]:
# YOUR CODE HERE
import re

def extract_numbers(text):
    return set(re.findall(r'\d+', text))

def has_hallucination(answer, sop):
    answer_nums = extract_numbers(answer)
    sop_nums    = extract_numbers(sop)
    unsupported = answer_nums - sop_nums
    return len(unsupported) > 0

def hallucination_rate(results, key):
    flagged = sum(has_hallucination(r[key], r["retrieved_sop"]) for r in results)
    return flagged / len(results) * 100

baseline_hallu = hallucination_rate(results, "baseline_output")
rag_hallu      = hallucination_rate(results, "rag_output")

print(f"Hallucination Frequency — Baseline : {baseline_hallu:.1f}%")
print(f"Hallucination Frequency — Naive RAG: {rag_hallu:.1f}%")

Hallucination Frequency — Baseline : 34.0%
Hallucination Frequency — Naive RAG: 28.0%


### **Task 3.4: Analyse Retrieval Impact**

#### **3.4.1 Compare Baseline and Solution V1 [4 marks]**
**The Task:** Quantify the impact of retrieval by comparing aggregate scores across Functional Correctness, Consistency, and Hallucination Frequency, with percentage changes.

**Hints & Tips:**
* Build a summary table: Baseline vs Naive RAG for each metric.
* Compute improvement percentages: `(rag - base) / base * 100`.
* Document WHERE retrieval helps and where it doesn't — both motivate Stage 4.

**Learner Inference:** This isolates retrieval's independent contribution before fine-tuning enters the picture.

In [15]:
# YOUR CODE HERE
import pandas as pd

summary = pd.DataFrame({
    "Metric": ["Format Adherence", "ROUGE-1", "ROUGE-L", "BLEU",
               "Consistency", "Hallucination Freq"],
    "Baseline": [baseline_fa, baseline_scores["rouge1"], baseline_scores["rougeL"],
                 baseline_scores["bleu"], 100.0, baseline_hallu],
    "Naive_RAG": [rag_fa, rag_scores["rouge1"], rag_scores["rougeL"],
                  rag_scores["bleu"], 100.0, rag_hallu],
    "HigherBetter": [True, True, True, True, True, False],
})

def improvement(base, rag, higher_better):
    if base == 0:
        return float("nan")            
    change = (rag - base) / base * 100
    return change if higher_better else -change   

summary["Improvement_%"] = summary.apply(
    lambda r: improvement(r["Baseline"], r["Naive_RAG"], r["HigherBetter"]),
    axis=1
)

summary = summary.drop(columns=["HigherBetter"])
print(summary.to_string(index=False))

            Metric   Baseline  Naive_RAG  Improvement_%
  Format Adherence 100.000000 100.000000       0.000000
           ROUGE-1   0.177233   0.191580       8.094629
           ROUGE-L   0.081020   0.103350      27.560016
              BLEU   0.002000   0.006404     220.188895
       Consistency 100.000000 100.000000       0.000000
Hallucination Freq  34.000000  28.000000      17.647059


---
## Save Artifacts

In [16]:
# YOUR CODE HERE
summary.to_csv("v1_metrics.csv", index=False)
print("Saved v1_metrics.csv")
print(summary.to_string(index=False))

Saved v1_metrics.csv
            Metric   Baseline  Naive_RAG  Improvement_%
  Format Adherence 100.000000 100.000000       0.000000
           ROUGE-1   0.177233   0.191580       8.094629
           ROUGE-L   0.081020   0.103350      27.560016
              BLEU   0.002000   0.006404     220.188895
       Consistency 100.000000 100.000000       0.000000
Hallucination Freq  34.000000  28.000000      17.647059


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 6.**

- [ ] ChromaDB reloaded from `./chroma_db/`
- [ ] **3.3.1** Automated testing run across full test set
- [ ] **3.3.2** Format adherence measured
- [ ] **3.3.3** ROUGE/BLEU computed with SOP-grounded references
- [ ] **3.3.4** Output consistency (determinism) verified
- [ ] **3.3.5** Hallucination frequency quantified
- [ ] **3.4.1** Retrieval impact quantified with improvement %
- [ ] **`v1_metrics.csv` saved** ← _Evidence for comparative analysis_

**If any item is unchecked, fix it before moving on.**